In [37]:
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, Input, Output

# load dataset
df = pd.read_csv(r"C:\Users\divya\Downloads\gapminder.csv")

# create app
app = Dash(__name__)

# KPI card style (define BEFORE layout)
card_style = {
    'background':'#f9f9f9',
    'padding':'20px',
    'borderRadius':'12px',
    'boxShadow':'0 4px 8px rgba(0,0,0,0.1)',
    'width':'30%',
    'textAlign':'center',
    'fontSize':'18px'
}

# layout
app.layout = html.Div([

    html.H1(
        "Global Development Dashboard",
        style={
            'textAlign':'center',
            'marginBottom':'25px'
        }
    ),

    # FILTERS
    html.Div([

        html.Div([
            html.Label("Select Continent"),
            dcc.Dropdown(
                id='continent',
                options=[{'label':i, 'value':i} for i in df['continent'].unique()],
                value=['Asia','Europe'],
                multi=True
            )
        ], style={'width':'35%'}),

        html.Div([
            html.Label("Select Year"),
            dcc.Slider(
                id='year',
                min=df['year'].min(),
                max=df['year'].max(),
                step=5,
                value=2007,
                marks={str(i):str(i) for i in df['year'].unique()}
            )
        ], style={'width':'55%'})

    ], style={'display':'flex','justifyContent':'space-between'}),

    html.Br(),

    # KPI CARDS
    html.Div([

        html.Div(id='kpi1', style=card_style),
        html.Div(id='kpi2', style=card_style),
        html.Div(id='kpi3', style=card_style)

    ], style={'display':'flex','justifyContent':'space-around'}),

    html.Br(),

    # GRAPH ROW 1
    html.Div([

        dcc.Graph(id='life_exp'),

        dcc.Graph(id='gdp_life')

    ], style={'display':'flex'}),

    # GRAPH ROW 2
    html.Div([

        dcc.Graph(id='population'),

        dcc.Graph(id='continent_dist')

    ], style={'display':'flex'})

])

# CALLBACK (controls interactivity)
@app.callback(

    Output('life_exp','figure'),
    Output('gdp_life','figure'),
    Output('population','figure'),
    Output('continent_dist','figure'),
    Output('kpi1','children'),
    Output('kpi2','children'),
    Output('kpi3','children'),

    Input('continent','value'),
    Input('year','value')

)
def update_dashboard(continent, year):

    # filter data
    dff = df[(df['continent'].isin(continent)) & (df['year']==year)]

    # GRAPH 1
    fig1 = px.line(
        df[df['continent'].isin(continent)],
        x='year',
        y='lifeExp',
        color='continent',
        title="Life Expectancy Trend"
    )

    # GRAPH 2
    fig2 = px.scatter(
        dff,
        x='gdpPercap',
        y='lifeExp',
        size='pop',
        color='continent',
        hover_name='country',
        log_x=True,
        title="GDP vs Life Expectancy"
    )

    # GRAPH 3
    fig3 = px.bar(
        dff,
        x='country',
        y='pop',
        color='continent',
        title="Population by Country"
    )

    # GRAPH 4
    fig4 = px.pie(
        dff,
        names='continent',
        values='pop',
        title="Population Distribution"
    )

    # KPI VALUES
    avg_life = round(dff['lifeExp'].mean(),1)
    total_pop = f"{dff['pop'].sum():,}"
    avg_gdp = round(dff['gdpPercap'].mean(),0)

    kpi1 = html.Div([
        html.H4("Avg Life Expectancy"),
        html.H2(avg_life)
    ])

    kpi2 = html.Div([
        html.H4("Total Population"),
        html.H2(total_pop)
    ])

    kpi3 = html.Div([
        html.H4("Avg GDP per Capita"),
        html.H2(avg_gdp)
    ])

    return fig1, fig2, fig3, fig4, kpi1, kpi2, kpi3

# function to open browser automatically
def open_browser():
    webbrowser.open_new("http://127.0.0.1:8050/")

# open browser after 1 second
threading.Timer(1, open_browser).start()
# run app
app.run(debug=True)

In [36]:
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc
import webbrowser
import threading

# load dataset
df = pd.read_csv(r"C:\Users\divya\Downloads\gapminder.csv")

# Graph 1
fig1 = px.line(
    df,
    x="year",
    y="lifeExp",
    color="continent",
    title="Life Expectancy over Years"
)

# Graph 2
fig2 = px.scatter(
    df,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    hover_name="country",
    log_x=True,
    title="GDP vs Life Expectancy"
)

# Graph 3
df_pop = df.groupby(["year","continent"])["pop"].sum().reset_index()

fig3 = px.bar(
    df_pop,
    x="year",
    y="pop",
    color="continent",
    title="Population Growth by Continent"
)

# Dash app
app = Dash(__name__)

app.layout = html.Div([

    html.H1(
        "Gapminder Dashboard",
        style={'textAlign':'center'}
    ),

    dcc.Graph(figure=fig1),

    dcc.Graph(figure=fig2),

    dcc.Graph(figure=fig3)

])

# function to open browser automatically
def open_browser():
    webbrowser.open_new("http://127.0.0.1:8050/")

# open browser after 1 second
threading.Timer(1, open_browser).start()

# run dash app
app.run(debug=True, use_reloader=False, port=8050)